In [1]:
import streamlit as st
import requests
import tempfile
import json

In [3]:
COMFYUI_API_URL = "http://127.0.0.1:8188/api/prompt"

st.set_page_config(page_title="Lyrics & Prompt to Audio via ComfyUI", page_icon="🎤")

st.title("🎤 Lyrics & Prompt to Audio via ComfyUI")
st.write("Enter lyrics and a prompt, send them to ComfyUI, and hear the generated audio.")

# Load your saved ComfyUI workflow JSON
with open("Ace_step_music_generation.json", "r") as f:
    workflow_data = json.load(f)

# Two separate inputs
lyrics = st.text_area("Enter lyrics:", height=150)
tags = st.text_input("Enter tags:")

if st.button("Generate Audio"):
    if not lyrics.strip() or not tags.strip():
        st.warning("Please enter both lyrics and tags.")
    else:
        try:
            # Update the workflow JSON with the user's inputs
            # Adjust these keys to match your actual workflow node IDs and input names
            workflow_data["6"]["inputs"]["lyrics"] = lyrics
            workflow_data["6"]["inputs"]["tags"] = tags

            with st.spinner("Generating audio..."):
                # Send the updated workflow to ComfyUI
                response = requests.post(COMFYUI_API_URL, json=workflow_data)
                response.raise_for_status()

                # ComfyUI returns JSON with output file info
                result = response.json()

                # Adjust these keys to match your output node ID
                audio_filename = result["10"]["inputs"]["filename_prefix"][0]["filename"]

                # Download the audio file from ComfyUI's /view endpoint
                audio_url = f"http://localhost:8188/view?filename={audio_filename}"
                audio_response = requests.get(audio_url)
                audio_response.raise_for_status()

                # Save audio to a temporary file
                temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                temp_file.write(audio_response.content)
                temp_file.close()

            st.success("Audio generated successfully!")
            st.audio(temp_file.name, format="audio/wav")

        except Exception as e:
            st.error(f"Error: {e}")

2025-09-20 12:38:51.493 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 12:38:51.494 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 12:38:51.540 
  command:

    streamlit run /home/avidmech/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-09-20 12:38:51.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 12:38:51.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 12:38:51.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 12:38:51.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor

In [1]:
import gradio as gr
import requests
import json
import tempfile
import time

COMFYUI_API_URL = "http://127.0.0.1:8188/prompt"
COMFYUI_HISTORY_URL = "http://127.0.0.1:8188/history"
COMFYUI_VIEW_URL = "http://127.0.0.1:8188/view"

# Load your saved ComfyUI workflow JSON
with open("workflow.json", "r") as f:
    workflow_nodes = json.load(f)

def generate_audio(lyrics, tags_prompt):
    # Make a copy so we don't overwrite the original
    wf = json.loads(json.dumps(workflow_nodes))

    # Update node 6 with user inputs
    wf["6"]["inputs"]["lyrics"] = lyrics
    wf["6"]["inputs"]["tags"] = tags_prompt

    # Wrap in {"prompt": ...}
    payload = {"prompt": wf}

    # Send to ComfyUI
    try:
        r = requests.post(COMFYUI_API_URL, json=payload)
        if r.status_code != 200:
            return None, f"Error from ComfyUI: {r.status_code} {r.text}"

        try:
            result = r.json()
        except Exception as e:
            return None, f"Invalid JSON from ComfyUI: {e} | Raw: {r.text}"

        prompt_id = result.get("prompt_id")
        if not prompt_id:
            return None, "No prompt_id returned from ComfyUI."

        # Poll for completion
        audio_filename = None
        for _ in range(60):  # wait up to ~60 seconds
            status = requests.get(f"{COMFYUI_HISTORY_URL}/{prompt_id}")
            if status.status_code == 200:
                history = status.json()
                if prompt_id in history:
                    outputs = history[prompt_id]["outputs"]
                    if "10" in outputs:  # Node 10 is SaveAudioMP3
                        audio_filename = outputs["10"]["inputs"]["filename_prefix"][0]
                        break
            time.sleep(1)

        if not audio_filename:
            return None, "Timed out waiting for audio file."

        # Download the audio file
        audio_url = f"{COMFYUI_VIEW_URL}?filename={audio_filename}"
        audio_data = requests.get(audio_url)
        if audio_data.status_code != 200:
            return None, f"Failed to download audio: {audio_data.status_code}"

        # Save to temp file
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        temp_file.write(audio_data.content)
        temp_file.close()

        return temp_file.name, "Audio generated successfully!"

    except Exception as e:
        return None, f"Error: {e}"

# Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🎤 Lyrics & Prompt to Audio via ComfyUI")

    with gr.Row():
        lyrics_input = gr.Textbox(label="Enter lyrics", lines=10, placeholder="Type your lyrics here...")
    with gr.Row():
        tags_input = gr.Textbox(label="Enter style/tags prompt", placeholder="e.g., rock, orchestral, energetic")

    with gr.Row():
        generate_btn = gr.Button("Generate Audio")

    with gr.Row():
        audio_output = gr.Audio(label="Generated Audio", type="filepath")
        status_output = gr.Textbox(label="Status / Messages")

    generate_btn.click(fn=generate_audio,
                       inputs=[lyrics_input, tags_input],
                       outputs=[audio_output, status_output])

if __name__ == "__main__":
    demo.launch()


FileNotFoundError: [Errno 2] No such file or directory: 'workflow.json'